# Lab 2 — Esplorare il dataset del TUG

## Prima di iniziare

**Il Timed Up and Go (TUG).** È una prova clinica semplice: la persona è seduta, si alza, cammina
per 3 metri, si gira, torna indietro e si siede. Si misura quanto tempo impiega, e serve a valutare
la mobilità e il rischio di caduta, soprattutto negli anziani (Podsiadlo e Richardson, J. Am.
Geriatr. Soc. 39(2), 1991). Con i sensori inerziali si può fare di più: riconoscere le singole fasi
e misurarne la durata.

**Il dataset** (Matey-Sanz, Casteleyn e Granell, Data in Brief 51, 2023; Zenodo 8398688): 23
soggetti che eseguono il TUG con uno smartphone nella tasca anteriore sinistra e uno smartwatch al
polso sinistro, sei canali ciascuno (accelerometro e giroscopio). Le etichette sono per campione,
cinque fasi: seduto, alzarsi, camminare, girarsi, sedersi. I confini fra le fasi sono letti sul video.

Oggi facciamo un'**esplorazione dei dati** (EDA) con **pandas**, la libreria Python delle tabelle.
Ogni parte fa una domanda sul dataset e la traduce in una tabella o in un grafico. I comandi di pandas
sono pochi e tornano in tutte le parti: la tabella qui sotto li raccoglie.

**Esercizi**: cinque «Scrivi tu» brevi. Ognuno è seguito da un controllo: se manca o è sbagliato, la cella si ferma
con un messaggio che dice che cosa ci si aspettava. Si va avanti solo dopo averlo sistemato.

**Teoria di riferimento** (Blocco 1.1): «One trial file, column by column», «The six columns of one
trial», «Device frame and body frame», «Timestamps of a real trial», «Loss, gaps and duplicates»,
«Cutting a trial into windows», «Which label does a window get?».

Il dataset serve da oggi. Prima di iniziare, da terminale nella radice del repository (circa 30 MB):

```
python scripts/get_data.py          # scarica e verifica l'MD5 (Zenodo 8398688)
python scripts/get_data.py --check  # 23 soggetti, 446 file
```

**Tempo**: prima di iniziare 10 min, Parte 1 15, Parte 2 15, Parte 3 15, Parte 4 20, Parte 5 15,
Parte 6 15, chiusura 5.

### Dove va questo laboratorio

| | domanda | output atteso |
|---|---|---|
| Parte 1 | Com'è fatta la tabella? | Righe, colonne, tipi, valori mancanti, statistiche dei sei canali. |
| Parte 2 | Chi è stato registrato, e quanto dura una prova? | Le tabelle dei soggetti e delle prove; la distribuzione delle durate. |
| Parte 3 | Come appare una prova nel tempo? | I segnali con le fasi, sul telefono e sull'orologio. |
| Parte 4 | Quanto pesa e quanto dura ogni fase? | Le fasi brevi (alzarsi, sedersi) sono rare: le classi sono sbilanciate. |
| Parte 5 | I campioni sono regolari e dentro il fondo scala? | Gli intervalli fra campioni dei due dispositivi; la quota di campioni saturi. |
| Parte 6 | Come si passa dalle prove alle finestre? | Finestre di 1 s con un'etichetta e una purezza; i due file nella cartella `work`. |

### Che cos'è un'EDA

L'**analisi esplorativa dei dati** (EDA, *exploratory data analysis*; J. W. Tukey, 1977) è il lavoro che si fa
**prima** di qualsiasi modello: guardare i dati con tabelle e grafici per capire com'è fatto il dataset, che cosa
contiene e che cosa c'è di strano. È un ciclo:

1. una **domanda** sui dati (le classi sono bilanciate? i campioni sono regolari?);
2. una **tabella o un grafico** che la traduce;
3. che cosa **si vede**: conteggi, durate, valori strani, buchi;
4. una **decisione scritta**, che vale per il resto del corso (ricampionare, dividere per soggetto, pesare le classi).

**Non è EDA**: addestrare un modello, togliere dati «brutti» senza scriverlo. Alla fine del notebook c'è la
tabella «Dall'EDA alle decisioni»: completala con quello che trovi.

### I comandi di pandas di oggi

Una tabella di pandas è un `DataFrame`; una sua colonna è una `Series`. `df` qui sotto è una tabella qualsiasi,
`s` una colonna. L'ultima colonna dice in quale slide del laboratorio il comando è spiegato con un esempio.

| comando | che cosa fa | slide |
|---|---|---|
| `df.head()`, `df.shape`, `df.dtypes` | le prime 5 righe; (righe, colonne); il tipo di ogni colonna | Gli strumenti: tabelle |
| `df.isna().sum()` | quanti valori mancano in ogni colonna | Gli strumenti: tabelle |
| `df.describe()`, `.round(2)` | conteggio, media, deviazione standard, minimo, quartili, massimo; arrotonda | Gli strumenti: tabelle |
| `df["col"]`, `df[["a", "b"]]` | una colonna; alcune colonne | pandas: colonne, righe, condizioni |
| `df[condizione]`, `(c1) & (c2)` | le righe dove la condizione è vera; due condizioni insieme | pandas: colonne, righe, condizioni |
| `s.value_counts()`, `normalize=True` | quante volte compare ogni valore; in quote invece che in conteggi | pandas: contare |
| `s.reindex(lista)` | mette le righe nell'ordine della lista | pandas: contare |
| `pd.crosstab(s1, s2)` | quante righe per ogni coppia di valori | pandas: contare |
| `df.groupby("a")["b"].mean()` | una statistica di `b` per ogni valore di `a` | groupby: dividere, calcolare, unire |
| `.size()`, `.agg(["min", "max"])`, `.describe()` | quante righe per gruppo; più statistiche insieme; tutte le statistiche | groupby: size, agg, describe |
| `df.loc[riga, colonna]`, `s.iloc[0]` | un valore per nome; un valore per posizione | groupby: size, agg, describe · pandas: trasformare colonne |
| `s.map(dizionario)` | sostituisce ogni valore con quello del dizionario | pandas: trasformare colonne |
| `s.diff()` | la differenza con la riga prima | pandas: trasformare colonne |
| `s == 0`, poi `.mean()` | una colonna di True e False; la sua media è la quota di True | pandas: trasformare colonne |
| `df.abs()`, `.any(axis=1)`, `s.clip(upper=40)` | valore assoluto; True se almeno una colonna della riga è True; taglia oltre 40 | pandas: trasformare colonne |
| `pd.concat([df1, df2])` | tabelle una sotto l'altra | pandas: unire tabelle |
| `df1.merge(df2, on="col")`, `.reset_index()` | unisce le righe con lo stesso valore di `col`; l'indice torna una colonna | pandas: unire tabelle |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from iomtlab.config import ALL_SUBJECTS, DATA_DIR, FULL_SCALE, LABEL_NAMES, LABEL_MAP, REPO_ROOT
from iomtlab.signal import list_executions, load_execution, load_subjects_info, load_executions_info, build_windows
from iomtlab.signal.windowing import label_window
from iomtlab.tug import phase_table
from iomtlab.features import extract
from iomtlab.io import save_features, save_windows

WORK = REPO_ROOT / "work"              # la cartella dei file che questo notebook scrive
WORK.mkdir(exist_ok=True)            # crea la cartella, se non c'è già
assert any(DATA_DIR.glob("s*/*.csv")), (
    "dataset assente: lancia `python scripts/get_data.py` dalla radice del repository")
if (DATA_DIR / "FAKE_DATA.txt").exists():   # il marcatore dei dati sintetici di make_fake_data.py
    print("ATTENZIONE: dati sintetici di prova; i valori non sono quelli veri.")

# tutte le prove dei due dispositivi in una sola tabella, una riga per campione
tabelle_prove = []                                 # qui metto una tabella per ogni prova e dispositivo
for soggetto in ALL_SUBJECTS:
    for dispositivo in ("sp", "sw"):               # sp: smartphone, sw: smartwatch
        for numero_prova in list_executions(soggetto, dispositivo):   # i numeri delle prove registrate, come testo: "01", "02", ...
            tabella_prova = load_execution(soggetto, numero_prova, dispositivo)   # una tabella: timestamp, sei canali, label
            tabella_prova["soggetto"] = soggetto       # quattro colonne nuove: da quale prova viene ogni riga
            tabella_prova["prova"] = int(numero_prova)
            tabella_prova["dispositivo"] = dispositivo
            tabella_prova["execution_id"] = f"{soggetto}_{numero_prova}"   # il nome della prova nel file delle prove, per esempio "s01_01"
            tabelle_prove.append(tabella_prova)
dati = pd.concat(tabelle_prove, ignore_index=True)         # le tabelle una sotto l'altra; ignore_index rinumera le righe da 0
print(len(tabelle_prove), "registrazioni,", len(dati), "righe")

## Parte 1 — Il primo sguardo alla tabella

> **Domanda.** Com'è fatta la tabella: quante righe, quali colonne, di che tipo, mancano valori?
> **Output atteso.** Le prime righe, i tipi, i valori mancanti e le statistiche dei sei canali.

Prima di qualsiasi grafico si guardano righe, colonne e tipi. Ogni riga di `dati` è un campione: il
`timestamp` in millisecondi, i tre assi dell'accelerometro (m/s²) e del giroscopio (rad/s), la fase scritta a mano
guardando il video (`label`) e le quattro colonne aggiunte qui sopra.

In [ ]:
display(dati.head())                    # head(): le prime 5 righe
print("righe e colonne:", dati.shape)   # shape: (righe, colonne)
print(dati.dtypes)                      # dtypes: il tipo di ogni colonna; int64 e float64 sono numeri, str o object è testo

In [ ]:
print(dati.isna().sum())                # isna(): True dove manca un valore; sum(): quanti True in ogni colonna
canali = ["x_acc", "y_acc", "z_acc", "x_gyro", "y_gyro", "z_gyro"]   # i sei canali dei due sensori
display(dati[canali].describe().round(2))   # una lista di nomi fra [] sceglie quelle colonne; round(2): due decimali

**Che cosa guardare.** Se mancano valori, e in quali colonne. Nelle medie dell'accelerometro: quale asse porta
la gravità (circa 9,8 m/s²)? Nei minimi e nei massimi: ci sono valori che sembrano impossibili?

### Scrivi tu — quante righe per dispositivo (circa 5 minuti)

In [ ]:
# 1. prendi la colonna "dispositivo" di dati: dati["dispositivo"]
# 2. conta quante volte compare ogni valore, con value_counts()
# 3. assegna il risultato a righe_per_dispositivo
righe_per_dispositivo = None

# controllo: i conteggi, sommati, devono dare tutte le righe della tabella
assert righe_per_dispositivo is not None, "da completare: scrivete righe_per_dispositivo"
assert righe_per_dispositivo.sum() == len(dati), "la somma dei conteggi non è il numero di righe: ricontrollate"
print(righe_per_dispositivo)
print("giusto")

## Parte 2 — I soggetti e le prove

> **Domanda.** Chi è stato registrato, quante prove ha fatto ognuno, quanto dura una prova?
> **Output atteso.** Le tabelle dei soggetti e delle prove; l'istogramma delle durate.

Il dataset ha due file di descrizione: uno con una riga per soggetto, uno con una riga per prova.

In [ ]:
soggetti = load_subjects_info()          # una riga per soggetto: età, genere, altezza, peso, mano dominante
display(soggetti.head())
display(soggetti.describe().round(1))    # describe usa solo le colonne numeriche
print(soggetti["gender"].value_counts())

In [ ]:
prove = load_executions_info()           # una riga per prova: verso del telefono in tasca, verso delle due svolte
display(prove.head())
print(prove["orientation"].value_counts())
# crosstab: quante prove per ogni coppia (prima svolta, seconda svolta); margins=True aggiunge i totali
display(pd.crosstab(prove["first_turn"], prove["second_turn"], margins=True))

**La durata di una prova** è l'ultimo timestamp meno il primo. `groupby` divide le righe in gruppi, qui uno per
prova, e `agg(["min", "max"])` calcola minimo e massimo di ogni gruppo: il risultato ha una riga per prova.

In [ ]:
telefono = dati[dati["dispositivo"] == "sp"]       # solo le righe del telefono
gruppi_per_prova = telefono.groupby(["soggetto", "prova"])   # i gruppi: uno per prova
durate_prove = gruppi_per_prova["timestamp"].agg(["min", "max"])         # di ogni gruppo il primo e l'ultimo timestamp
durate_prove["durata_s"] = (durate_prove["max"] - durate_prove["min"]) / 1000   # da millisecondi a secondi
display(durate_prove.head())
print(durate_prove["durata_s"].describe().round(1))

plt.hist(durate_prove["durata_s"], bins=20)          # istogramma: 20 barre, quante prove per ogni fascia di durata
plt.xlabel("durata della prova [s]")
plt.ylabel("numero di prove")
plt.show()

### Scrivi tu — quante prove per soggetto (circa 5 minuti)

In [ ]:
# 1. durate_prove ha una riga per prova, e il suo indice ha due livelli: soggetto e prova
# 2. conta le righe di ogni soggetto: durate_prove.groupby("soggetto").size()
# 3. assegna il risultato a prove_per_soggetto
prove_per_soggetto = None

# controllo: sommate, le prove per soggetto devono dare tutte le prove del telefono
assert prove_per_soggetto is not None, "da completare: scrivete prove_per_soggetto"
assert prove_per_soggetto.sum() == len(durate_prove), "la somma non è il numero di prove: ricontrollate"
print(prove_per_soggetto.describe().round(1))   # quante prove per soggetto: media, minimo, massimo
print("giusto")

**Che cosa guardare.** L'età: il TUG nasce per gli anziani, ma chi è stato registrato qui? Quante prove fa ogni
persona: le prove dello stesso soggetto si somigliano, e questo conterà quando divideremo i dati in addestramento
e verifica. Le durate: quanto dura una prova, e ci sono prove molto più lunghe o più corte delle altre?

## Parte 3 — Una prova nel tempo

> **Domanda.** Come appare una prova, fase per fase, sul telefono e sull'orologio?
> **Output atteso.** Tre assi di un sensore nel tempo, con sotto la fase di ogni campione.

Per scegliere le righe di una prova servono tre condizioni insieme, unite da `&` («e»), ognuna fra parentesi.
La funzione qui sotto le usa e disegna la prova con matplotlib.

In [ ]:
NOMI_DISPOSITIVI = {"sp": "telefono", "sw": "orologio"}

def disegna_prova(soggetto, numero_prova, dispositivo, sensore="acc"):
    # le righe di una prova: tre condizioni unite da &, ognuna fra parentesi
    prova_singola = dati[(dati["soggetto"] == soggetto) & (dati["prova"] == numero_prova) & (dati["dispositivo"] == dispositivo)]
    tempo_s = (prova_singola["timestamp"] - prova_singola["timestamp"].iloc[0]) / 1000   # il tempo in secondi dall'inizio; iloc[0]: il primo valore

    # due grafici, uno sopra l'altro, con lo stesso asse dei tempi; quello sopra è alto il triplo
    fig, (grafico_segnale, grafico_fase) = plt.subplots(2, 1, sharex=True, figsize=(9, 5), height_ratios=[3, 1])
    for asse in ["x", "y", "z"]:
        colonna = f"{asse}_{sensore}"                        # per esempio "x_acc"
        grafico_segnale.plot(tempo_s, prova_singola[colonna], label=asse)
    grafico_segnale.set_ylabel("m/s²" if sensore == "acc" else "rad/s")
    grafico_segnale.set_title(f"{soggetto}, prova {numero_prova}, {NOMI_DISPOSITIVI[dispositivo]}")
    grafico_segnale.legend(loc="upper right")

    numero_fase = prova_singola["label"].map(LABEL_MAP)                       # map: ogni etichetta diventa il suo numero (SEATED 0, STANDING_UP 1, ...)
    grafico_fase.plot(tempo_s, numero_fase)
    grafico_fase.set_yticks(range(len(LABEL_NAMES)), LABEL_NAMES)   # sull'asse y i nomi delle fasi al posto dei numeri
    grafico_fase.set_xlabel("tempo [s]")
    plt.show()

In [ ]:
disegna_prova("s01", 1, "sp")            # telefono, accelerometro
disegna_prova("s01", 1, "sw")            # orologio, stessa prova
disegna_prova("s01", 1, "sp", "gyro")    # telefono, giroscopio

**Che cosa guardare.** Da seduti le linee sono piatte e c'è solo la gravità. Alzandosi e sedendosi
l'accelerometro del telefono cambia livello, perché il telefono in tasca si inclina. Camminando si vedono
oscillazioni al ritmo dei passi. Nelle svolte un asse del giroscopio prende un valore grande. Il polso si muove
con il braccio: la camminata e le svolte si vedono bene, l'alzarsi e il sedersi meno che sul telefono.
Prova con un'altra prova o un altro soggetto: basta cambiare gli argomenti della funzione.

## Parte 4 — Le fasi in tutto il dataset

> **Domanda.** Quanto pesa ogni fase nel dataset, e quanto dura?
> **Output atteso.** Le fasi brevi (alzarsi, sedersi) sono una piccola parte dei campioni: le classi sono sbilanciate.

`value_counts(normalize=True)` dà le frazioni invece dei conteggi.

In [ ]:
quote_fasi = telefono["label"].value_counts(normalize=True) * 100   # la quota di campioni di ogni fase, in percentuale
quote_fasi = quote_fasi.reindex(LABEL_NAMES)    # reindex: le fasi nell'ordine della prova, non dalla più frequente
print(quote_fasi.round(1))

plt.bar(quote_fasi.index, quote_fasi.values)       # una barra per fase: i nomi sotto (index), le quote in altezza (values)
plt.ylabel("campioni del telefono [%]")
plt.xticks(rotation=30)                  # nomi inclinati di 30 gradi, per non sovrapporli
plt.show()

**La durata delle fasi.** Una prova ha otto fasi in fila (seduto, alzarsi, camminare, girarsi, camminare, girarsi,
sedersi, seduto). La funzione `phase_table` della libreria trasforma una prova in una tabella con una riga per fase:
`label`, inizio `t0` e fine `t1` in ms, durata `dur_s` in secondi, numero di campioni `n`. Un `groupby` dentro un
`for` dà, a ogni giro, la chiave del gruppo e le sue righe.

In [ ]:
tabelle_fasi = []                                   # qui metto la tabella delle fasi di ogni prova
for (soggetto, numero_prova), prova_singola in gruppi_per_prova:   # a ogni giro: la chiave (soggetto, prova) e le sue righe
    fasi_prova = phase_table(prova_singola)         # una riga per fase della prova
    fasi_prova["soggetto"] = soggetto
    tabelle_fasi.append(fasi_prova)
fasi = pd.concat(tabelle_fasi, ignore_index=True)   # le tabelle delle fasi una sotto l'altra, come per dati
display(fasi.head(8))                    # le fasi della prima prova, in ordine

durate_fasi = fasi.groupby("label")["dur_s"].describe().reindex(LABEL_NAMES)   # le statistiche della durata, una riga per fase
display(durate_fasi.round(2))

# un box per fase: la linea è la mediana, la scatola va dal primo al terzo quartile
plt.boxplot([fasi[fasi["label"] == nome]["dur_s"] for nome in LABEL_NAMES])
plt.xticks(range(1, len(LABEL_NAMES) + 1), LABEL_NAMES, rotation=30)   # le scatole stanno alle posizioni 1, 2, 3, ...
plt.ylabel("durata della fase [s]")
plt.show()

### Scrivi tu — il numero di oggi (circa 5 minuti)

La durata mediana, in secondi, della fase «camminare» sul telefono.

In [ ]:
# 1. tieni solo le righe di fasi in cui label è "WALKING": fasi[fasi["label"] == "WALKING"]
# 2. di quelle righe prendi la colonna "dur_s" e la sua mediana, con median()
# 3. assegna il risultato a mediana
mediana = None

# controllo: la mediana è il 50 % di describe; .loc[riga, colonna] prende un valore della tabella durate_fasi
assert mediana is not None, "da completare: scrivete mediana"
atteso = durate_fasi.loc["WALKING", "50%"]
assert abs(mediana - atteso) < 1e-9, f"risultato {mediana:.2f}, atteso {atteso:.2f} (la colonna 50% di durate_fasi)"
print(f"giusto: durata mediana di una fase di camminata {mediana:.2f} s")

**Lo stesso gesto, quattro versi del telefono.** Il telefono è in tasca con lo schermo verso la gamba o verso
fuori, dritto o capovolto: il verso di ogni prova è nella colonna `orientation` della tabella `prove`. Da seduti
l'accelerometro misura solo la gravità, e la sua media sull'asse x dice come stava il telefono. `merge` unisce due
tabelle sulle righe che hanno lo stesso valore in una colonna, qui `execution_id`.

In [ ]:
campioni_seduti = telefono[telefono["label"] == "SEATED"]                       # solo i campioni da seduti
x_seduti = campioni_seduti.groupby("execution_id")["x_acc"].mean().reset_index()   # la media di x_acc per prova; reset_index: execution_id torna una colonna
x_seduti = x_seduti.merge(prove, on="execution_id")   # aggiunge a ogni prova le colonne di prove: orientation, first_turn, second_turn
display(x_seduti.head())
display(x_seduti.groupby("orientation")["x_acc"].agg(["size", "mean", "min", "max"]).round(2))   # size: quante prove

**Che cosa guardare.** Camminare e stare seduti occupano quasi tutti i campioni; alzarsi e sedersi durano un
secondo o due e sono una piccola parte del dataset. Un classificatore che dice sempre «camminare» avrebbe
un'accuratezza alta senza riconoscere le fasi brevi. Il segno della media di x cambia con il verso del telefono:
un valore che dipende dal segno di un asse cambia da un soggetto all'altro, la norma del vettore no.

## Parte 5 — La qualità dei campioni

> **Domanda.** I campioni arrivano a intervalli regolari? I valori restano dentro il fondo scala?
> **Output atteso.** Le statistiche degli intervalli fra campioni sui due dispositivi; la quota di campioni saturi.

`diff()` sottrae a ogni valore quello della riga prima. Fatto dentro un `groupby` per prova, non sottrae l'ultimo
campione di una prova dal primo della successiva: la prima riga di ogni prova resta senza intervallo (NaN).

In [ ]:
# l'intervallo fra ogni campione e il precedente, in ms, prova per prova
gruppi_per_registrazione = dati.groupby(["soggetto", "prova", "dispositivo"])   # uno per prova e dispositivo
dati["dt_ms"] = gruppi_per_registrazione["timestamp"].diff()                    # diff dentro ogni gruppo: la prima riga di ogni prova resta NaN
display(dati.groupby("dispositivo")["dt_ms"].describe().round(1))   # describe salta i NaN

fig, grafici = plt.subplots(1, 2, figsize=(9, 3), sharey=True)        # due grafici affiancati
for grafico, dispositivo in zip(grafici, ["sp", "sw"]):                   # zip: a ogni giro un grafico e un dispositivo
    intervalli = dati[dati["dispositivo"] == dispositivo]["dt_ms"]      # gli intervalli di quel dispositivo
    grafico.hist(intervalli.clip(upper=40), bins=41, range=(0, 41))            # clip: gli intervalli oltre 40 ms finiscono nell'ultima barra
    grafico.set_yscale("log")                                         # scala logaritmica: si vedono anche le barre piccole
    grafico.set_title(NOMI_DISPOSITIVI[dispositivo])
    grafico.set_xlabel("intervallo fra campioni [ms]")
grafici[0].set_ylabel("numero di intervalli")
plt.show()

### Scrivi tu — gli intervalli ripetuti (circa 5 minuti)

Un intervallo di 0 ms vuol dire due campioni con lo stesso timestamp: sono arrivati insieme, in un gruppo.

In [ ]:
# 1. crea in dati la colonna "ripetuto": True dove dt_ms è uguale a 0, cioè dati["dt_ms"] == 0
# 2. raggruppa dati per "dispositivo" e prendi la media della colonna "ripetuto":
#    la media di True e False è la quota di True (True vale 1, False vale 0)
# 3. moltiplica per 100 e assegna il risultato a ripetuti_pct
ripetuti_pct = None

# controllo: una quota per dispositivo, sp e sw, fra 0 e 100
assert ripetuti_pct is not None, "da completare: scrivete ripetuti_pct"
assert sorted(ripetuti_pct.index) == ["sp", "sw"], "servono due quote, una per sp e una per sw"
assert 0 <= ripetuti_pct.min() and ripetuti_pct.max() <= 100, "le quote devono stare fra 0 e 100: avete moltiplicato per 100?"
print(ripetuti_pct.round(1))
print("giusto")

**La saturazione.** Un sensore misura fino al suo fondo scala: oltre, il valore resta fermo al massimo. Contiamo
i campioni con almeno un asse al 99 % del fondo scala o oltre.

In [ ]:
righe_saturazione = []                                # una riga della tabella finale per ogni coppia (dispositivo, sensore)
for dispositivo in ["sp", "sw"]:
    righe_dispositivo = dati[dati["dispositivo"] == dispositivo]   # solo le righe di quel dispositivo
    for sensore in ["acc", "gyro"]:
        colonne = [f"x_{sensore}", f"y_{sensore}", f"z_{sensore}"]
        fondo_scala = FULL_SCALE[dispositivo][sensore]      # il fondo scala, in m/s² o rad/s
        # abs(): il valore assoluto; any(axis=1): True se almeno un asse della riga supera la soglia
        campione_saturo = (righe_dispositivo[colonne].abs() >= 0.99 * fondo_scala).any(axis=1)
        righe_saturazione.append({"dispositivo": dispositivo, "sensore": sensore,
                      "fondo scala": round(fondo_scala, 1), "campioni saturi %": round(campione_saturo.mean() * 100, 3)})
display(pd.DataFrame(righe_saturazione))                          # una lista di dizionari diventa una tabella, una riga per dizionario

**Che cosa guardare.** La mediana degli intervalli dice a che frequenza lavora ogni dispositivo; la deviazione
standard, il massimo e la quota di intervalli a 0 ms dicono quanto è regolare. Confronta telefono e orologio. Se
gli intervalli non sono tutti uguali, prima di tagliare finestre bisogna ricampionare su una griglia regolare.

## Parte 6 — Dalle prove alle finestre

> **Domanda.** Come si passa dalle prove alle finestre?
> **Output atteso.** Finestre di 1 s con un'etichetta e una purezza; i due file nella cartella `work`.

Un classificatore prende in ingresso blocchi di lunghezza fissa. Ogni prova si ricampiona a 50 Hz e si taglia in
**finestre** di 50 campioni (1 s), con un passo di 25 campioni (0,5 s). Una finestra che sta a cavallo di due fasi
riceve la classe della **maggioranza** dei suoi campioni; la **purezza** è la quota di campioni di quella classe.

### Scrivi tu — l'etichetta di una finestra (circa 10 minuti)

**Che cosa entra nella funzione.** `classi_campioni` è un array numpy di **50 interi**, uno per ogni campione della
finestra, in ordine di tempo. Ogni intero è la classe di quel campione: 0 SEATED, 1 STANDING_UP, 2 WALKING, 3 TURNING,
4 SITTING_DOWN. La cella qui sotto mostra una finestra vera: quella da 1,5 a 2,5 s della prova s01_01 sul telefono,
ricampionata a 50 Hz. Comincia con 28 campioni di alzarsi e finisce con 22 di camminata.

**Che cosa esce.** Una tupla di due valori: la classe più frequente nella finestra, e `True` se quella classe copre
almeno l'80 % dei campioni. Per la finestra d'esempio: `(1, False)`, perché vince alzarsi ma con 28 / 50 = 0,56.

In [ ]:
# un esempio vero: la finestra da 1,5 a 2,5 s della prova s01_01, telefono, ricampionata a 50 Hz
esempio = np.array([1] * 28 + [2] * 22)    # 28 campioni di alzarsi (classe 1), poi 22 di camminata (classe 2)
print(esempio)                              # i 50 interi, in ordine di tempo
print(len(esempio), "campioni")

In [ ]:
def etichetta_finestra(classi_campioni, min_purezza=0.8):
    # classi_campioni: un array di 50 interi, la classe di ogni campione della finestra (0 SEATED … 4 SITTING_DOWN)
    # 1. conta quante volte compare ogni classe: np.bincount(classi_campioni) dà un conteggio per ogni intero da 0 in su
    # 2. la classe di maggioranza è quella con il conteggio più alto: np.argmax(conteggi)
    #    (a parità argmax prende la prima, cioè la classe più piccola)
    # 3. la purezza è il conteggio di quella classe diviso il numero di campioni della finestra
    # 4. restituisci una tupla: (classe, purezza >= min_purezza)
    raise NotImplementedError

In [ ]:
# sei finestre di 50 campioni, con il risultato atteso: cinque scritte a mano e quella vera di s01_01
casi = [
    (np.array([2] * 50), (2, True)),                # tutta camminata: pura
    (np.array([0] * 30 + [1] * 20), (0, False)),    # 60 % seduto: vince seduto, ma non è pura
    (np.array([3] * 45 + [2] * 5), (3, True)),      # 90 % girarsi: pura
    (np.array([4] * 40 + [0] * 10), (4, True)),     # 80 % esatto: pura, perché la soglia è «almeno 0,8»
    (np.array([1] * 25 + [0] * 25), (0, False)),    # parità 25 e 25: vince la classe più piccola
    (np.array([1] * 28 + [2] * 22), (1, False)),    # la finestra vera di s01_01: 56 % alzarsi, non pura
]
for classi_campioni, atteso in casi:
    risultato = etichetta_finestra(classi_campioni)
    assert risultato == atteso, f"atteso {atteso}, risultato {risultato}"
print("giusto: i sei casi tornano")

**Tutte le finestre del telefono.** `build_windows` della libreria fa per tutte le prove quello che abbiamo appena
descritto: ricampiona a 50 Hz, taglia le finestre, dà a ognuna la classe di maggioranza e la purezza. Il risultato
è un dizionario di array; lo trasformiamo in una tabella per contare con pandas.

In [ ]:
finestre_telefono = build_windows("sp", size=50, shift=25)   # tutte le prove del telefono: finestre di 1 s, passo 0,5 s
# finestre_telefono è un dizionario: X le finestre (finestre × 50 campioni × 6 canali), y la classe, purity la purezza, subject il soggetto
print(len(finestre_telefono["y"]), "finestre di forma", finestre_telefono["X"].shape[1:])

tabella_finestre = pd.DataFrame({"classe": np.array(LABEL_NAMES)[finestre_telefono["y"]],   # dal numero della classe al suo nome
                         "pura": finestre_telefono["purity"] >= 0.8})
finestre_per_classe = tabella_finestre.groupby("classe")["pura"].agg(["size", "mean"]).reindex(LABEL_NAMES)   # size: quante finestre; mean: la quota di pure
finestre_per_classe.columns = ["finestre", "quota pure"]                     # nomi italiani per le due colonne
display(finestre_per_classe.round(2))

n_pure = finestre_per_classe["finestre"] * finestre_per_classe["quota pure"]            # il numero di finestre pure di ogni classe
plt.bar(finestre_per_classe.index, n_pure, label="pure (purezza ≥ 0,8)")
plt.bar(finestre_per_classe.index, finestre_per_classe["finestre"] - n_pure, bottom=n_pure, label="ambigue")   # bottom: la barra parte sopra la prima
plt.ylabel("numero di finestre")
plt.xticks(rotation=30)
plt.legend()
plt.show()

In [ ]:
# le feature di ogni finestra (una riga per finestra) e i loro nomi
feature, nomi_feature = extract(finestre_telefono["X"], blocks=("amplitude", "dispersion", "dynamics", "norm"))
print(feature.shape[1], "feature per finestra")
# i due file servono da qui in avanti: non cancellare la cartella work
print("scritto", save_features(WORK / "features.npz", feature, finestre_telefono["y"], nomi_feature, finestre_telefono["subject"]))
print("scritto", save_windows(WORK / "windows.npz", finestre_telefono))

**Che cosa guardare.** Le fasi brevi, l'alzarsi e il sedersi, durano poco più di una finestra: hanno poche
finestre, e una parte di quelle è ambigua, perché contiene anche la fase prima o quella dopo.

## Chiusura — Dall'EDA alle decisioni

Quello che l'EDA trova diventa una scelta per tutto il corso. Completa la terza colonna con i numeri che hai risultato.

| che cosa abbiamo visto | che cosa decidiamo | il tuo numero |
|---|---|---|
| timestamp irregolari, alcuni ripetuti (Parte 5) | ricampionare tutto a 50 Hz | intervalli ripetuti sul telefono: … % |
| una decina di prove per persona (Parte 2) | dividere i dati per soggetto, mai per finestra | prove per soggetto (mediana): … |
| classi sbilanciate (Parte 4) | misurare con la macro-F1, non con l'accuratezza | quota della classe più rara: … % |
| fasi brevi: poche finestre, molte ambigue (Parte 6) | una soglia di purezza; finestre di 1 s, passo 0,5 s | quota di finestre «alzarsi» pure: … |
| telefono in tasca in quattro versi (Parte 4) | guardare anche la norma, che non dipende dal verso | prove per verso (Parte 2): … |
| saturazione (Parte 5) | tenere i valori grezzi, senza tagli | campioni saturi: … % |

**Da riportare**

1. La durata mediana della fase di camminata, in secondi.
2. Una frase: quale classe è più rara nel dataset, e perché sarà un problema per un classificatore.